# 🎙️ RAG-Driven Meeting Intelligence System with BERT Embeddings and Open-Source LLMs

This notebook analyzes real meeting transcripts from the **AMI Corpus** using:
- **BERT embeddings** (via `sentence-transformers`) for semantic search
- **FAISS** as a fast vector store
- **FLAN-T5-Large** as a free, open-source generative LLM
- **RAG (Retrieval-Augmented Generation)** to ground answers in transcript evidence


---
**Enable GPU before running:** `Runtime → Change runtime type → T4 GPU`

## Cell 1 — Install Dependencies

Install all required libraries. This may take 2–3 minutes on first run.
- `transformers` + `accelerate` + `bitsandbytes` — for loading and quantizing LLMs
- `datasets` — to load the AMI Corpus from HuggingFace
- `faiss-cpu` — fast vector similarity search
- `sentence-transformers` — BERT-based semantic embeddings

In [ ]:
!pip install -q transformers datasets faiss-cpu sentence-transformers \
               accelerate bitsandbytes pandas numpy torch

print("✅ All dependencies installed.")

## Cell 2 — Imports and GPU Check

Import all libraries and verify that a CUDA GPU is available.  
If you see `CPU` instead of a GPU name, go to `Runtime → Change runtime type → T4 GPU`.

In [ ]:
import torch
import numpy as np
import pandas as pd
import faiss
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from collections import defaultdict

# ── GPU / CPU detection ──────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = "cuda"
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = "cpu"
    print("⚠️  No GPU found — running on CPU (will be slow).")
    print("   Go to Runtime → Change runtime type → T4 GPU for best performance.")

print(f"\nUsing device: {device}")

## Cell 3 — Load AMI Corpus

Load the **AMI Meeting Corpus** from HuggingFace Datasets.
- Dataset: `edinburghcstr/ami`, config `ihm` (individual headset microphone), split `test`
- We group transcript segments into:
  - `meeting_transcripts`: full transcript per meeting
  - `speaker_transcripts`: per-speaker transcript per meeting

In [ ]:
print("Loading AMI Corpus from HuggingFace (this may take a minute)...")

# Load the test split of the AMI corpus (IHM = individual headset microphone)
dataset = load_dataset("edinburghcstr/ami", "ihm", split="test", trust_remote_code=True)

print(f"✅ Loaded {len(dataset)} transcript segments.")
print(f"   Columns: {dataset.column_names}")

# ── Group segments by meeting_id ─────────────────────────────────────────────
meeting_transcripts = defaultdict(list)   # meeting_id → [segment text, ...]
speaker_transcripts = defaultdict(lambda: defaultdict(list))  # meeting_id → speaker_id → [text]

for row in dataset:
    meeting_id = row.get("meeting_id", "unknown")
    speaker_id = row.get("speaker_id", "unknown")
    text       = row.get("text", "").strip()

    if not text:          # skip empty segments
        continue

    meeting_transcripts[meeting_id].append(text)
    speaker_transcripts[meeting_id][speaker_id].append(text)

# Join segments into single strings
meeting_transcripts = {mid: " ".join(segs) for mid, segs in meeting_transcripts.items()}
speaker_transcripts = {
    mid: {sid: " ".join(segs) for sid, segs in speakers.items()}
    for mid, speakers in speaker_transcripts.items()
}

# ── Preview ──────────────────────────────────────────────────────────────────
meeting_ids = sorted(meeting_transcripts.keys())
print(f"\n📋 Available meetings ({len(meeting_ids)} total):")
for mid in meeting_ids:
    speakers = list(speaker_transcripts[mid].keys())
    word_count = len(meeting_transcripts[mid].split())
    print(f"  {mid} — {len(speakers)} speakers, ~{word_count:,} words")
    print(f"    Speakers: {', '.join(speakers)}")

## Cell 4 — Load Embedding Model (BERT via SentenceTransformer)

Load `all-MiniLM-L6-v2`, a lightweight and fast BERT-based model that produces 384-dimensional sentence embeddings.

We also define a `chunk_text()` function that splits long transcripts into overlapping word windows so that no context is lost at chunk boundaries.

In [ ]:
print("Loading sentence embedding model...")
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embed_model = embed_model.to(device)
print("✅ Embedding model loaded.")

# ── Chunking helper ──────────────────────────────────────────────────────────
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50) -> list[str]:
    """
    Split `text` into overlapping word-level chunks.
    chunk_size : number of words per chunk
    overlap    : number of words shared between consecutive chunks
    """
    words  = text.split()
    chunks = []
    step   = chunk_size - overlap          # how far to advance each iteration

    for start in range(0, len(words), step):
        chunk = " ".join(words[start : start + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
        if start + chunk_size >= len(words):  # reached end
            break

    return chunks if chunks else [text]    # always return at least one chunk


# ── Embed all chunks for all meetings ────────────────────────────────────────
all_chunks_metadata = []   # list of dicts: {embedding, text, meeting_id, speaker_id}

print("\nChunking and embedding transcripts...")
for meeting_id, speakers in speaker_transcripts.items():
    for speaker_id, transcript in speakers.items():
        if not transcript.strip():
            continue

        chunks = chunk_text(transcript, chunk_size=200, overlap=50)

        # Batch-encode all chunks for this speaker at once (faster than one-by-one)
        embeddings = embed_model.encode(
            chunks,
            batch_size=64,
            show_progress_bar=False,
            convert_to_numpy=True
        )

        for chunk_text_str, emb in zip(chunks, embeddings):
            all_chunks_metadata.append({
                "embedding"  : emb,
                "text"       : chunk_text_str,
                "meeting_id" : meeting_id,
                "speaker_id" : speaker_id,
            })

print(f"✅ Created {len(all_chunks_metadata):,} embedded chunks across {len(meeting_transcripts)} meetings.")

## Cell 5 — Build FAISS Index

Stack all embeddings into a single NumPy matrix and build a **FAISS `IndexFlatL2`** index for fast nearest-neighbour retrieval.

FAISS performs exhaustive L2 (Euclidean) search — perfect for our dataset size and gives exact results.

In [ ]:
# Stack all embeddings into a single float32 matrix  [N × 384]
embedding_matrix = np.vstack([c["embedding"] for c in all_chunks_metadata]).astype("float32")

embedding_dim = embedding_matrix.shape[1]   # 384 for MiniLM
print(f"Embedding matrix shape: {embedding_matrix.shape}")

# Build a flat (exact) L2 index
faiss_index = faiss.IndexFlatL2(embedding_dim)
faiss_index.add(embedding_matrix)

print(f"✅ FAISS index built with {faiss_index.ntotal:,} vectors.")

## Cell 6 — Load Open-Source LLM (FLAN-T5-Large)

Load **`google/flan-t5-large`** — an instruction-tuned encoder-decoder model that:
- Fits comfortably in ~4 GB of GPU VRAM
- Requires **no HuggingFace login**
- Is well-suited for Q&A and summarization tasks

We also define `generate_answer()` which takes a prompt string and returns the model's generated response.

In [ ]:
MODEL_NAME = "google/flan-t5-large"

print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model (this may take ~1 minute on first download)...")
llm_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",          # automatically maps layers to available GPU
)

print(f"✅ {MODEL_NAME} loaded on {device}.")


# ── Generation function ──────────────────────────────────────────────────────
def generate_answer(prompt: str, max_new_tokens: int = 300) -> str:
    """
    Generate a text response for the given prompt using FLAN-T5.

    Parameters
    ----------
    prompt         : The full instruction + context string.
    max_new_tokens : Maximum number of tokens in the generated response.

    Returns
    -------
    str : Decoded text response.
    """
    # Tokenise — truncate very long prompts to fit the model's 512-token input limit
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    ).to(device)

    # Generate
    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,          # beam search for better quality
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    # Decode and strip special tokens
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


# Quick sanity check
test_out = generate_answer("Summarize in one sentence: Meetings are used to discuss project progress and assign tasks.")
print(f"\n🧪 Sanity check output: {test_out}")

---
### 💡 Want a Stronger LLM?

FLAN-T5-Large is great for Colab free tier, but if you have a HuggingFace account and access to more VRAM, consider upgrading:

| Model | VRAM needed | Notes |
|---|---|---|
| `mistralai/Mistral-7B-Instruct-v0.2` | ~8 GB (with 4-bit quant) | Excellent instruction following; requires HF login + model access |
| `meta-llama/Llama-3.2-3B` | ~5 GB (with 4-bit quant) | Strong and compact; requires HF login + Meta licence |

To use them, replace the `MODEL_NAME` in Cell 6 and add:
```python
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
llm_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
```
You will also need to adjust `generate_answer()` to use `AutoModelForCausalLM` and handle the causal LM output format.

---

## Cell 7 — RAG Retrieval Function

Define `retrieve_chunks()` which:
1. Encodes the query with the same BERT embedding model
2. Searches FAISS for the top-K nearest chunks
3. Optionally filters by `meeting_id` and/or `speaker_id` so results are always relevant

In [ ]:
def retrieve_chunks(
    query: str,
    meeting_id: str | None = None,
    speaker_id: str | None = None,
    top_k: int = 5,
    fetch_k: int = 50,          # fetch extra candidates before filtering
) -> list[str]:
    """
    Retrieve the top-K most semantically relevant transcript chunks for a query.

    Parameters
    ----------
    query      : Natural language question or search string.
    meeting_id : If provided, restrict results to this meeting.
    speaker_id : If provided, restrict results to this speaker.
    top_k      : Number of chunks to return after filtering.
    fetch_k    : Number of FAISS candidates to retrieve before filtering.

    Returns
    -------
    list[str] : The matched text chunks (up to top_k).
    """
    # Step 1 — embed the query
    query_embedding = embed_model.encode(
        [query], convert_to_numpy=True, show_progress_bar=False
    ).astype("float32")

    # Step 2 — search FAISS for top fetch_k candidates
    distances, indices = faiss_index.search(query_embedding, fetch_k)
    candidate_indices  = indices[0]   # shape (fetch_k,)

    # Step 3 — apply metadata filters
    results = []
    for idx in candidate_indices:
        if idx < 0 or idx >= len(all_chunks_metadata):  # safety check
            continue

        meta = all_chunks_metadata[idx]

        # Filter by meeting_id if requested
        if meeting_id and meta["meeting_id"] != meeting_id:
            continue

        # Filter by speaker_id if requested
        if speaker_id and meta["speaker_id"] != speaker_id:
            continue

        results.append(meta["text"])

        if len(results) >= top_k:
            break

    # Edge case: no results found after filtering
    if not results:
        print(f"  ⚠️  No chunks found for meeting={meeting_id}, speaker={speaker_id}.")

    return results

## Cell 8 — Core RAG + LLM Q&A Function

Define `answer_query()`, the main pipeline that:
1. Retrieves relevant chunks via FAISS
2. Assembles them into a context string
3. Constructs a clean instruction prompt
4. Calls FLAN-T5 to generate a grounded answer

In [ ]:
def answer_query(
    query: str,
    meeting_id: str,
    speaker_id: str | None = None,
    top_k: int = 5,
) -> str:
    """
    Full RAG pipeline: retrieve → prompt → generate.

    Parameters
    ----------
    query      : The question to answer.
    meeting_id : The meeting to query against.
    speaker_id : Optional — restrict context to a specific speaker.
    top_k      : Number of retrieved chunks to include in context.

    Returns
    -------
    str : The LLM-generated answer.
    """
    # Step 1 — retrieve semantically relevant chunks
    chunks = retrieve_chunks(
        query, meeting_id=meeting_id, speaker_id=speaker_id, top_k=top_k
    )

    # Edge case: no context found
    if not chunks:
        return "No relevant transcript segments found for this query."

    # Step 2 — join chunks into a context passage
    context = " [...] ".join(chunks)   # separator marks chunk boundaries

    # Step 3 — build the instruction prompt
    prompt = (
        "Based on the following meeting transcript excerpts, answer this question.\n"
        f"Question: {query}\n"
        f"Context: {context}\n"
        "Answer:"
    )

    # Step 4 — generate and return the answer
    return generate_answer(prompt, max_new_tokens=300)

## Cell 9 — Predefined Query Functions

Six high-level wrapper functions for common meeting-analysis tasks.  
Each one calls `answer_query()` with a task-specific prompt template.

In [ ]:
def get_meeting_topic(meeting_id: str) -> str:
    """Identify the main topic or agenda of a meeting."""
    query = "What was the main topic or agenda of this meeting?"
    return answer_query(query, meeting_id=meeting_id)


def summarize_speaker(meeting_id: str, speaker_id: str) -> str:
    """Summarise everything a specific speaker said in a meeting."""
    # Restrict FAISS retrieval to only this speaker's chunks
    query = f"Summarize everything that speaker {speaker_id} said in this meeting."
    return answer_query(query, meeting_id=meeting_id, speaker_id=speaker_id)


def get_action_items(meeting_id: str) -> str:
    """Extract action items, decisions, and next steps from a meeting."""
    query = "What action items, decisions, or next steps were discussed in this meeting?"
    return answer_query(query, meeting_id=meeting_id)


def get_meeting_summary(meeting_id: str) -> str:
    """Generate a detailed overall summary of a meeting."""
    query = "Provide a detailed summary of the entire meeting."
    # Use more chunks for a comprehensive summary
    return answer_query(query, meeting_id=meeting_id, top_k=8)


def get_speaker_sentiment(meeting_id: str, speaker_id: str) -> str:
    """Analyse the tone and sentiment of a specific speaker."""
    query = f"What was the tone and sentiment of speaker {speaker_id} during the meeting?"
    return answer_query(query, meeting_id=meeting_id, speaker_id=speaker_id)


def compare_speakers(meeting_id: str) -> str:
    """Compare the contributions and roles of all speakers in a meeting."""
    query = "How did the different speakers contribute to the meeting? Compare their roles."
    return answer_query(query, meeting_id=meeting_id, top_k=8)


print("✅ All 6 query functions defined.")

## Cell 10 — Demo Run

Run all 6 analysis functions on the first available meeting in the dataset.  
Results are printed with clear section headers for readability.

⏱️ *Each query takes ~5–15 seconds on a T4 GPU.*

In [ ]:
import time

# ── Pick the first available meeting ─────────────────────────────────────────
demo_meeting_id = meeting_ids[0]
demo_speakers   = sorted(speaker_transcripts[demo_meeting_id].keys())
demo_speaker_1  = demo_speakers[0] if demo_speakers else None

print("=" * 60)
print(f"DEMO MEETING: {demo_meeting_id}")
print(f"SPEAKERS: {', '.join(demo_speakers)}")
print("=" * 60)


def print_section(title: str, content: str):
    """Pretty-print a labelled result section."""
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    print(content)


# ── 1. Meeting Topic ─────────────────────────────────────────────────────────
t0 = time.time()
topic = get_meeting_topic(demo_meeting_id)
print_section("1. MEETING TOPIC", topic)
print(f"   ⏱ {time.time()-t0:.1f}s")


# ── 2. Speaker Summary ───────────────────────────────────────────────────────
if demo_speaker_1:
    t0 = time.time()
    spk_summary = summarize_speaker(demo_meeting_id, demo_speaker_1)
    print_section(f"2. SPEAKER SUMMARY — {demo_speaker_1}", spk_summary)
    print(f"   ⏱ {time.time()-t0:.1f}s")
else:
    print("\n⚠️  No speakers found — skipping speaker summary.")


# ── 3. Action Items ──────────────────────────────────────────────────────────
t0 = time.time()
actions = get_action_items(demo_meeting_id)
print_section("3. ACTION ITEMS & DECISIONS", actions)
print(f"   ⏱ {time.time()-t0:.1f}s")


# ── 4. Full Meeting Summary ──────────────────────────────────────────────────
t0 = time.time()
summary = get_meeting_summary(demo_meeting_id)
print_section("4. FULL MEETING SUMMARY", summary)
print(f"   ⏱ {time.time()-t0:.1f}s")


# ── 5. Speaker Sentiment ─────────────────────────────────────────────────────
if demo_speaker_1:
    t0 = time.time()
    sentiment = get_speaker_sentiment(demo_meeting_id, demo_speaker_1)
    print_section(f"5. SPEAKER SENTIMENT — {demo_speaker_1}", sentiment)
    print(f"   ⏱ {time.time()-t0:.1f}s")
else:
    print("\n⚠️  No speakers found — skipping sentiment analysis.")


# ── 6. Speaker Comparison ────────────────────────────────────────────────────
t0 = time.time()
comparison = compare_speakers(demo_meeting_id)
print_section("6. SPEAKER COMPARISON", comparison)
print(f"   ⏱ {time.time()-t0:.1f}s")


print("\n" + "=" * 60)
print("✅ DEMO COMPLETE")
print("=" * 60)

---

## ➕ Interactive Query (Optional)

Use the cell below to run your own custom questions against any meeting or speaker:

In [ ]:
# ── Customise these ──────────────────────────────────────────────────────────
custom_meeting_id = meeting_ids[0]        # change to any meeting ID
custom_speaker_id = None                  # set to a speaker ID or leave None
custom_query      = "What technical challenges were mentioned in the meeting?"

# ── Run ──────────────────────────────────────────────────────────────────────
print(f"Meeting : {custom_meeting_id}")
print(f"Speaker : {custom_speaker_id or 'All speakers'}")
print(f"Query   : {custom_query}\n")

result = answer_query(custom_query, meeting_id=custom_meeting_id, speaker_id=custom_speaker_id)
print("Answer:")
print(result)